# Crypto Price Prediction - Project Demo
**Student:** Jean Trochet  
**Course:** Advanced Programming 2025

This notebook demonstrates the full Machine Learning pipeline: Data Collection, Feature Engineering, Training, and Backtesting.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path so we can import our modules
sys.path.append(os.path.abspath('../src'))

from data_loader import fetch_crypto_data
from feature_engineer import add_technical_indicators
from models import train_rf_model
from evaluation import Backtester

## 1. Fetch Data
We fetch the last 5 years of Bitcoin data from Yahoo Finance.

In [ ]:
coin = "bitcoin"
csv_path = fetch_crypto_data(coin, days=1825)
print(f"Data saved to: {csv_path}")

df = pd.read_csv(csv_path, parse_dates=True, index_col='timestamp')
df['price'].plot(title=f"{coin.upper()} Price History", figsize=(10, 5))
plt.show()

## 2. Feature Engineering
We calculate RSI, Bollinger Bands, and Moving Averages.

In [ ]:
df_features = add_technical_indicators(df)
df_features[['price', 'bb_upper', 'bb_lower']].tail(100).plot(figsize=(10, 5), title="Bollinger Bands (Last 100 Days)")
plt.show()

## 3. Model Training & Backtest
We use a Random Forest Classifier with GridSearch to predict price direction.

In [ ]:
model, X_test, y_test, predictions = train_rf_model(df_features)

# Run Backtest
test_prices = df_features.loc[X_test.index, 'price']
backtester = Backtester(initial_balance=10000)
final_balance = backtester.run(pd.DataFrame({'price': test_prices, 'prediction': predictions}, index=X_test.index))

print(f"Final Portfolio Value: ${final_balance.iloc[-1]['value']:.2f}")
backtester.plot_results(pd.DataFrame({'price': test_prices}), filename='../results/demo_chart.png')